### Gathering Historical Fantasy Scores

In the cells below I will be gathering historical fantasy football players points for a specified season. I will first look at the 2025 season and perform transformations, cleaning, and EDA to determine if more data is needed and identify holes. This data is useful because we can use the historical points as a way to compare out models output performance as in the accuracy of our output.

This data is gathered from ESPN's default-leage endpoint api: [Fantasy API 2025](https://lm-api-reads.fantasy.espn.com/apis/v3/games/ffl/seasons/2025)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import os
#root directory of project
root = Path.cwd().parent
data_path = root/"data"

In [2]:
def write_to_dict(df, name):
    '''Update dictionary csv files '''
    columns = df.columns.to_numpy()
    dict_ = pd.DataFrame(columns)
    try:
        dict_.to_csv(data_path/f"dicts/{name}_dictionary.csv")
    except Exception as e:
        print(f"failed to write dictionary with exception {e}")

def update_data(df, name):
    try:
        df.to_csv(data_path/f"{name}_final.csv")
    except Exception as e:
        print(f"failed to updata data: {e}")

In [3]:
## load fantasy historical data
weekly_df = pd.read_csv(data_path/"player_weeks_2025.csv")
weekly_df.head()


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
0,4429795,Jahmyr Gibbs,RB,8,1,15.0,18.419261
1,3117251,Christian McCaffrey,RB,25,1,23.2,17.690153
2,4362628,Ja'Marr Chase,WR,4,1,4.6,19.783577
3,4426515,Puka Nacua,WR,14,1,23.1,16.765436
4,4430807,Bijan Robinson,RB,1,1,24.4,19.748249


In [4]:
# load player summary data
player_sum = pd.read_csv(data_path/"player_summary_2025.csv")
player_sum.head()

,player_id,player,position,team,games,avg_opp_def_rank,total_rushingAttempts,total_rushingYards,total_rushingTouchdowns,total_receptions,...,avg_fieldGoalPct,avg_fieldGoalsMadeYardsAverage,avg_avgInterceptionYards,avg_fantasy_points,longRushing,longReception,longPassing,longFieldGoalMade,longInterception,total_fantasy_points
0,3117251,Christian McCaffrey,RB,SF,17,16.000000,311.0,1202.0,10.0,102.0,...,NaN,NaN,NaN,24.505882,41.0,39.0,NaN,NaN,NaN,416.60
1,4426515,Puka Nacua,WR,LAR,16,15.750000,10.0,105.0,1.0,129.0,...,NaN,NaN,NaN,23.437500,45.0,58.0,NaN,NaN,NaN,375.00
2,4430807,Bijan Robinson,RB,ATL,17,16.529412,287.0,1478.0,7.0,79.0,...,NaN,NaN,NaN,21.811765,93.0,69.0,NaN,NaN,NaN,370.80
3,4429795,Jahmyr Gibbs,RB,DET,17,16.882353,243.0,1223.0,13.0,77.0,...,NaN,NaN,NaN,21.582353,78.0,42.0,NaN,NaN,NaN,366.90
4,3918298,Josh Allen,QB,BUF,17,16.470588,112.0,579.0,14.0,NaN,...,NaN,NaN,NaN,21.448235,40.0,NaN,54.0,NaN,NaN,364.62


In [5]:
write_to_dict(player_sum, "player_sum")


In [6]:
# load player log data
player_logs = pd.read_csv(data_path/"player_gamelogs_2025.csv")
player_logs.head()


,player_id,player,position,team,season,season_type,week,eventId,date,opponent,...,fieldGoalsMade40_49,fieldGoalAttempts40_49,fieldGoalsMade50,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points
0,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23,20.3
1,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,6.4
2,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,22.8
3,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,9.8
4,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32,37.0


In [7]:
write_to_dict(player_logs, "player_logs")

In [8]:
display(weekly_df.shape)
display(player_sum.shape)
display(player_logs.shape)

(13958, 7)

(685, 104)

(7572, 70)

In [9]:
display(weekly_df["player_id"].nunique())
display(player_sum["player_id"].nunique())
display(player_logs["player_id"].nunique())

1088

685

685

In [10]:
## match our historical fantasy players with our other data
players_in_sum = player_sum["player_id"].unique()
players_set = set(players_in_sum)

players_in_logs = player_logs["player_id"].unique()
players_set_2 = set(players_in_logs)

assert players_set == players_set_2, "not same players"

In [11]:
# now that two of our df contain the same players, we only need to filter the historical fantasy data to contain the player_id as the other dataframes

weekly_df = weekly_df[weekly_df["player_id"].isin(players_set)]
weekly_df.shape

(11356, 7)

## cleaning missing values in projected points

There are some cases where a player was not set to play but ended up earning fantasy points and hence missing values for the projected points. So if a player has missing projected points and scored fantasy points, I am going to impute the projected points with either the median or mean of their fantasy points of that season, I will use the player summary dataframe to calculate these statistics and then determine which of the central tendencies I would like to use

In [12]:
mask = weekly_df[weekly_df["projected_points"].isna() & weekly_df["fantasy_points"].isna()]

# check why there are no columns with both no point and no projected points
no_proj = weekly_df[weekly_df["projected_points"].isna()]
no_pts = weekly_df[weekly_df["fantasy_points"].isna()]

display(mask.head())
display(no_proj.head())
display(no_pts.head())

print("length of no projected points:", len(no_proj))
print("length of no proj with week = 1:", len(no_proj[no_proj["week"] == 1]))
print(len(no_pts[no_pts["projected_points"] == 0]))

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
355,4601080,Raheim Sanders,RB,5,1,6.3,NaN
422,4430656,Myles Price,WR,16,1,0.0,NaN
450,4715355,Darius Cooper,WR,21,1,0.0,NaN
453,5160110,Jahdae Walker,WR,3,1,0.0,NaN
742,4876006,Zaire Mitchell-Paden,TE,18,1,0.0,NaN


,player_id,player,position,pro_team_id,week,fantasy_points,projected_points
144,4240657,Michael Carter,RB,22,1,NaN,0.0
149,4574716,Harrison Mevis,K,14,1,NaN,0.0
150,4430968,Kimani Vidal,RB,24,1,NaN,0.0
167,4034949,Eddy Pineiro,K,25,1,NaN,0.0
227,4569682,Audric Estime,RB,18,1,NaN,0.0


length of no projected points: 703
length of no proj with week = 1: 8
1064


In [13]:
# in player summary we already have average fantasy points so I will compute the mean 

weekly_df["med_fantasy_pts"]  = weekly_df.groupby("player_id")["fantasy_points"].transform("median")
weekly_df.head()

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points,med_fantasy_pts
0,4429795,Jahmyr Gibbs,RB,8,1,15.0,18.419261,19.4
1,3117251,Christian McCaffrey,RB,25,1,23.2,17.690153,24.1
2,4362628,Ja'Marr Chase,WR,4,1,4.6,19.783577,19.9
3,4426515,Puka Nacua,WR,14,1,23.1,16.765436,22.8
4,4430807,Bijan Robinson,RB,1,1,24.4,19.748249,21.2


In [14]:
weekly_df["projected_points"] = np.where(
    weekly_df["projected_points"].isna(),
    weekly_df["med_fantasy_pts"],
    weekly_df["projected_points"]
)

weekly_df.head()

,player_id,player,position,pro_team_id,week,fantasy_points,projected_points,med_fantasy_pts
0,4429795,Jahmyr Gibbs,RB,8,1,15.0,18.419261,19.4
1,3117251,Christian McCaffrey,RB,25,1,23.2,17.690153,24.1
2,4362628,Ja'Marr Chase,WR,4,1,4.6,19.783577,19.9
3,4426515,Puka Nacua,WR,14,1,23.1,16.765436,22.8
4,4430807,Bijan Robinson,RB,1,1,24.4,19.748249,21.2


# Data Integration
From looking at the Data dictionaries, our summary and player logs contain the fantasy data already so I will Merge the logs with the ```weekly_df``` on player_id and week

In [15]:
#key columns are columns we merge on
key_cols = {"player_id","week"}

#find set of columns that overlap excluding key columns
overlapping_cols = (set(player_logs.columns) & set(weekly_df.columns)) - key_cols


#overlapping_cols

#left merge on key columns, drop overlapping columns
merged_df = pd.merge(player_logs, weekly_df.drop(columns=list(overlapping_cols)), on=list(key_cols), how="left")

merged_df.head()

,player_id,player,position,team,season,season_type,week,eventId,date,opponent,...,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,pro_team_id,projected_points,med_fantasy_pts
0,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,...,NaN,NaN,NaN,NaN,NaN,23,20.3,8,21.873502,19.4
1,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,...,NaN,NaN,NaN,NaN,NaN,7,6.4,8,20.787567,19.4
2,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,...,NaN,NaN,NaN,NaN,NaN,17,22.8,8,23.848417,19.4
3,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,...,NaN,NaN,NaN,NaN,NaN,10,9.8,8,21.707558,19.4
4,4429795,Jahmyr Gibbs,RB,DET,2025,2025 Regular Season,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,...,NaN,NaN,NaN,NaN,NaN,32,37.0,8,22.944125,19.4


In [16]:
merged_df["team"].nunique()
merged_df["season_type"].nunique()

if merged_df["season_type"].nunique() == 1:
    merged_df = merged_df.drop(columns = ["pro_team_id", "season_type"])
#we have team name and team ID number
# merged_df.drop(columns = "pro_team_id")

merged_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,fieldGoalsMade50,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,projected_points,med_fantasy_pts
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,@,...,NaN,NaN,NaN,NaN,NaN,NaN,23,20.3,21.873502,19.4
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,@,...,NaN,NaN,NaN,NaN,NaN,NaN,7,6.4,20.787567,19.4
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,vs,...,NaN,NaN,NaN,NaN,NaN,NaN,17,22.8,23.848417,19.4
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,@,...,NaN,NaN,NaN,NaN,NaN,NaN,10,9.8,21.707558,19.4
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,vs,...,NaN,NaN,NaN,NaN,NaN,NaN,32,37.0,22.944125,19.4


## Adding in Weather Data
I would also like to add weather data to this dataset, so i want to add a location tag based on the team name abbreviatoin and correlated city. From ChatGPT I was able to genereate a dictionary which contains the teamname abbreviations as the keys, and the city, State as the values, from this I will cross reference the atVs attribute which tells us if the game is at home or away and then using that information I will declare the location to be at home as in the teams name corresponding location or at the oppontent teams location in a ```location``` column

In [17]:
## lets create a location column instead of opponent and atV's
# nfl teams 
nfl_teams = {
    "ARI": "Glendale, AZ",
    "ATL": "Atlanta, GA",
    "BAL": "Baltimore, MD",
    "BUF": "Orchard Park, NY",
    "CAR": "Charlotte, NC",
    "CHI": "Chicago, IL",
    "CIN": "Cincinnati, OH",
    "CLE": "Cleveland, OH",
    "DAL": "Arlington, TX",
    "DEN": "Denver, CO",
    "DET": "Detroit, MI",
    "GB": "Green Bay, WI",
    "HOU": "Houston, TX",
    "IND": "Indianapolis, IN",
    "JAX": "Jacksonville, FL",
    "KC": "Kansas City, MO",
    "LV": "Paradise, NV",
    "LAC": "Inglewood, CA",
    "LAR": "Inglewood, CA",
    "MIA": "Miami Gardens, FL",
    "MIN": "Minneapolis, MN",
    "NE": "Foxborough, MA",
    "NO": "New Orleans, LA",
    "NYG": "East Rutherford, NJ",
    "NYJ": "East Rutherford, NJ",
    "PHI": "Philadelphia, PA",
    "PIT": "Pittsburgh, PA",
    "SF": "Santa Clara, CA",
    "SEA": "Seattle, WA",
    "TB": "Tampa, FL",
    "TEN": "Nashville, TN",
    "WAS": "Landover, MD"
}

dict = {"vs": "home", "@": "away"}
merged_df["atVs"] = merged_df["atVs"].map(dict)
merged_df.head()

#put a location of game instead of decyphering three columns
# merged_df["location"] = np.where(
#     merged_df["atVs"] == "home",
#     merged_df["team"],
#     merged_df["opponent"])

# merged_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,fieldGoalsMade50,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,projected_points,med_fantasy_pts
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,NaN,NaN,NaN,NaN,NaN,NaN,23,20.3,21.873502,19.4
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,NaN,NaN,NaN,NaN,NaN,NaN,7,6.4,20.787567,19.4
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,NaN,NaN,NaN,NaN,NaN,NaN,17,22.8,23.848417,19.4
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,NaN,NaN,NaN,NaN,NaN,NaN,10,9.8,21.707558,19.4
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,NaN,NaN,NaN,NaN,NaN,NaN,32,37.0,22.944125,19.4


In [18]:
#put a location of game instead of decyphering three columns
merged_df["location"] = np.where(
    merged_df["atVs"] == "home",
    merged_df["team"],
    merged_df["opponent"])

merged_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,projected_points,med_fantasy_pts,location
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,NaN,NaN,NaN,NaN,NaN,23,20.3,21.873502,19.4,CHI
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,NaN,NaN,NaN,NaN,NaN,7,6.4,20.787567,19.4,MIN
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,NaN,NaN,NaN,NaN,NaN,17,22.8,23.848417,19.4,DET
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,NaN,NaN,NaN,NaN,NaN,10,9.8,21.707558,19.4,LAR
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,NaN,NaN,NaN,NaN,NaN,32,37.0,22.944125,19.4,DET


In [19]:
# finally map location to an actual place
merged_df["location"] = merged_df["location"].map(nfl_teams)

merged_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,projected_points,med_fantasy_pts,location
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,NaN,NaN,NaN,NaN,NaN,23,20.3,21.873502,19.4,"Chicago, IL"
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,NaN,NaN,NaN,NaN,NaN,7,6.4,20.787567,19.4,"Minneapolis, MN"
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,NaN,NaN,NaN,NaN,NaN,17,22.8,23.848417,19.4,"Detroit, MI"
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,NaN,NaN,NaN,NaN,NaN,10,9.8,21.707558,19.4,"Inglewood, CA"
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,NaN,NaN,NaN,NaN,NaN,32,37.0,22.944125,19.4,"Detroit, MI"


In [20]:
merged_df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,fieldGoalAttempts50,fieldGoalsMade,fieldGoalAttempts,extraPointsMade,extraPointAttempts,opp_def_rank,fantasy_points,projected_points,med_fantasy_pts,location
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,NaN,NaN,NaN,NaN,NaN,23,20.3,21.873502,19.4,"Chicago, IL"
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,NaN,NaN,NaN,NaN,NaN,7,6.4,20.787567,19.4,"Minneapolis, MN"
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,NaN,NaN,NaN,NaN,NaN,17,22.8,23.848417,19.4,"Detroit, MI"
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,NaN,NaN,NaN,NaN,NaN,10,9.8,21.707558,19.4,"Inglewood, CA"
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,NaN,NaN,NaN,NaN,NaN,32,37.0,22.944125,19.4,"Detroit, MI"


In [21]:
merged_df["player"].info()


<class 'pandas.Series'>
RangeIndex: 7572 entries, 0 to 7571
Series name: player
Non-Null Count  Dtype
--------------  -----
7572 non-null   str  
dtypes: str(1)
memory usage: 156.2 KB


## Adding in Weather Data
Using ```nflready``` library, we can gather the weather location based on the value in the ```eventId``` column and use that to append corresponding weather data to the rows. 

In [22]:
import nflreadpy as nfl
import polars as pl


sched = nfl.load_schedules(seasons=2025)  
weather = sched.select([
    "espn", "game_id", "gameday", "gametime",
    "stadium", "roof", "surface", "temp", "wind"
])
weather = weather.to_pandas()
# your_df: your ESPN data with an event ID column
merged_df["eventId"] = merged_df["eventId"].astype(str)
df = merged_df.merge(weather, left_on="eventId", right_on="espn", how="left")
df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,location,espn,game_id,gameday,gametime,stadium,roof,surface,temp,wind
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,"Chicago, IL",401772965,2025_18_DET_CHI,2026-01-04,16:25,Soldier Field,outdoors,grass,33.0,4.0
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,"Minneapolis, MN",401772711,2025_17_DET_MIN,2025-12-25,16:30,U.S. Bank Stadium,dome,sportturf,NaN,NaN
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,"Detroit, MI",401772806,2025_16_PIT_DET,2025-12-21,16:25,Ford Field,dome,fieldturf,NaN,NaN
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,"Inglewood, CA",401772909,2025_15_DET_LA,2025-12-14,16:25,SoFi Stadium,dome,matrixturf,NaN,NaN
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,"Detroit, MI",401772947,2025_14_DAL_DET,2025-12-04,20:15,Ford Field,dome,fieldturf,NaN,NaN


### Updating temperature data
We are going to impute the temperature missing values for domes to be the average temperature of a football dome stadium which happens to be around 70 farhenhiet.

In [23]:
df["temp"] = np.where(
    df["roof"] == "dome",
    70,
    df["temp"])

df.head()

,player_id,player,position,team,season,week,eventId,date,opponent,atVs,...,location,espn,game_id,gameday,gametime,stadium,roof,surface,temp,wind
0,4429795,Jahmyr Gibbs,RB,DET,2025,18,401772965,2026-01-04T21:25:00.000+00:00,CHI,away,...,"Chicago, IL",401772965,2025_18_DET_CHI,2026-01-04,16:25,Soldier Field,outdoors,grass,33.0,4.0
1,4429795,Jahmyr Gibbs,RB,DET,2025,17,401772711,2025-12-25T21:30:00.000+00:00,MIN,away,...,"Minneapolis, MN",401772711,2025_17_DET_MIN,2025-12-25,16:30,U.S. Bank Stadium,dome,sportturf,70.0,NaN
2,4429795,Jahmyr Gibbs,RB,DET,2025,16,401772806,2025-12-21T21:25:00.000+00:00,PIT,home,...,"Detroit, MI",401772806,2025_16_PIT_DET,2025-12-21,16:25,Ford Field,dome,fieldturf,70.0,NaN
3,4429795,Jahmyr Gibbs,RB,DET,2025,15,401772909,2025-12-14T21:25:00.000+00:00,LAR,away,...,"Inglewood, CA",401772909,2025_15_DET_LA,2025-12-14,16:25,SoFi Stadium,dome,matrixturf,70.0,NaN
4,4429795,Jahmyr Gibbs,RB,DET,2025,14,401772947,2025-12-05T01:15:00.000+00:00,DAL,home,...,"Detroit, MI",401772947,2025_14_DAL_DET,2025-12-04,20:15,Ford Field,dome,fieldturf,70.0,NaN


In [24]:
df = df.drop(columns = "date")


In [25]:
## update our data file
write_to_dict(df, "df")
update_data(df, "df")